## Tumor Subtype Classification using MOFA factors

In [21]:
import pandas as pd
import mofax as mfx

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, balanced_accuracy_score, precision_score, recall_score, f1_score, average_precision_score

In [22]:
def load_mofa_factors(model_path, clinical_index):
    model = mfx.mofa_model(model_path)
    factors = model.get_factors(df = True)
    factors = factors.loc[clinical_index]
    return factors

In [23]:
mofa_models = {
    "mofa_trained_lg2": "../../data/latent/mofa_trained_lg2.hdf5",
    "mofa_trained_vsn": "../../data/latent/mofa_trained_vsn.hdf5",
    "mofa_trained_lg2_fs": "../../data/latent/mofa_trained_lg2_fs.hdf5",
    "mofa_trained_vsn_fs": "../../data/latent/mofa_trained_vsn_fs.hdf5"
}

In [24]:
clinical_data = pd.read_csv('../../data/cleaned_data/clinical_cleaned.csv', index_col = 0)
y = clinical_data['tumor_subtype'].map({'other_subtype': 1, 'ductal_type': 0})

In [25]:
X = load_mofa_factors(mofa_models['mofa_trained_lg2'], clinical_data.index)

Stratified Split

In [26]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 42, stratify = y)

#### Training

In [9]:
scoring = {
    'balanced_accuracy': make_scorer(balanced_accuracy_score),
    'precision': make_scorer(precision_score, zero_division = 0),
    'recall': make_scorer(recall_score, zero_division = 0),
    'f1': make_scorer(f1_score, zero_division = 0),
    'pr_auc_score': make_scorer(average_precision_score, response_method='predict_proba')
}

In [10]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [2, 3, 4, 5],
    'min_samples_leaf': [5, 10, 15, 20],
    'class_weight': ['balanced', 'balanced_subsample']
}

In [11]:
rf = RandomForestClassifier(random_state = 42)

GridSearchCV with Stratified 5-Fold Cross Validation

In [12]:
cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)
grid_search_rf = GridSearchCV(estimator = rf, param_grid = param_grid, cv = cv, scoring = scoring, refit = 'pr_auc_score', n_jobs = -1, return_train_score = True)
grid_search_rf.fit(X_train, y_train)

,estimator,RandomForestC...ndom_state=42)
,param_grid,"{'class_weight': ['balanced', 'balanced_subsample'], 'max_depth': [2, 3, ...], 'min_samples_leaf': [5, 10, ...], 'n_estimators': [50, 100, ...]}"
,scoring,"{'balanced_accuracy': make_scorer(b...hod='predict'), 'f1': make_scorer(f...ro_division=0), 'pr_auc_score': make_scorer(a...redict_proba'), 'precision': make_scorer(p...ro_division=0), ...}"
,n_jobs,-1
,refit,'pr_auc_score'
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,True
,n_estimators,50


Get the best model

In [13]:
best_model = grid_search_rf.best_estimator_
print("Best hyperparameters:", grid_search_rf.best_params_)

Best hyperparameters: {'class_weight': 'balanced_subsample', 'max_depth': 2, 'min_samples_leaf': 5, 'n_estimators': 50}


Get train and validation scores of best_estimator

In [14]:
best_index = grid_search_rf.best_index_
cv_results = grid_search_rf.cv_results_

In [15]:
train_metrics = {
    'balanced_accuracy': cv_results['mean_train_balanced_accuracy'][best_index],
    'precision': cv_results['mean_train_precision'][best_index],
    'recall': cv_results['mean_train_recall'][best_index],
    'f1': cv_results['mean_train_f1'][best_index],
    'pr_auc_score': cv_results['mean_train_pr_auc_score'][best_index]
}

In [16]:
val_metrics = {
    'balanced_accuracy': cv_results['mean_test_balanced_accuracy'][best_index],
    'precision': cv_results['mean_test_precision'][best_index],
    'recall': cv_results['mean_test_recall'][best_index],
    'f1': cv_results['mean_test_f1'][best_index],
    'pr_auc_score': cv_results['mean_test_pr_auc_score'][best_index]
}

#### Testing

In [17]:
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1] # Probability for the 'other_subtype' (class 1)

In [18]:
test_metrics = {
    'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred, zero_division = 0),
    'recall': recall_score(y_test, y_pred, zero_division = 0),
    'f1': f1_score(y_test, y_pred, zero_division = 0),
    'pr_auc_score': average_precision_score(y_test, y_prob)
}

test_metrics

{'balanced_accuracy': 0.8166666666666667,
 'precision': 0.8,
 'recall': 0.6666666666666666,
 'f1': 0.7272727272727273,
 'pr_auc_score': 0.5326388888888889}

Results

In [19]:
results_df = pd.DataFrame([train_metrics, val_metrics, test_metrics],
                          index = ['Train', 'Validation', 'Test'])
results_df = results_df.T

In [20]:
display(results_df)

,Train,Validation,Test
balanced_accuracy,0.756410,0.607428,0.816667
precision,0.723810,0.416667,0.800000
recall,0.555556,0.300000,0.666667
f1,0.626960,0.342063,0.727273
pr_auc_score,0.785778,0.450789,0.532639


SVM

In [27]:
scoring = {
    'balanced_accuracy': make_scorer(balanced_accuracy_score),
    'precision': make_scorer(precision_score, zero_division = 0),
    'recall': make_scorer(recall_score, zero_division = 0),
    'f1': make_scorer(f1_score, zero_division = 0),
    'pr_auc_score': make_scorer(average_precision_score, response_method='decision_function'),
}

In [28]:
param_grid_svm = {'svm__C': [0.01, 0.1, 1, 10, 100],}

In [29]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', LinearSVC(class_weight='balanced', dual = "auto", random_state=42, max_iter=10000)),])

In [30]:
cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)
grid_search_svm = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid_svm,
    scoring=scoring,
    refit="pr_auc_score",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)
grid_search_svm.fit(X_train, y_train)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'svm__C': [0.01, 0.1, ...]}"
,scoring,"{'balanced_accuracy': make_scorer(b...hod='predict'), 'f1': make_scorer(f...ro_division=0), 'pr_auc_score': make_scorer(a...ion_function'), 'precision': make_scorer(p...ro_division=0), ...}"
,n_jobs,-1
,refit,'pr_auc_score'
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,True
,copy,True


In [31]:
best_model = grid_search_svm.best_estimator_
print("Best hyperparameters:", grid_search_svm.best_params_)

Best hyperparameters: {'svm__C': 0.01}


In [32]:
best_index = grid_search_svm.best_index_
cv_results = grid_search_svm.cv_results_

In [33]:
train_metrics = {
    'balanced_accuracy': cv_results['mean_train_balanced_accuracy'][best_index],
    'precision': cv_results['mean_train_precision'][best_index],
    'recall': cv_results['mean_train_recall'][best_index],
    'f1': cv_results['mean_train_f1'][best_index],
    'pr_auc_score': cv_results['mean_train_pr_auc_score'][best_index]
}

In [34]:
val_metrics = {
    'balanced_accuracy': cv_results['mean_test_balanced_accuracy'][best_index],
    'precision': cv_results['mean_test_precision'][best_index],
    'recall': cv_results['mean_test_recall'][best_index],
    'f1': cv_results['mean_test_f1'][best_index],
    'pr_auc_score': cv_results['mean_test_pr_auc_score'][best_index]
}

In [35]:
y_pred = best_model.predict(X_test)
y_score = best_model.decision_function(X_test)

In [36]:
test_metrics = {
    'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred, zero_division = 0),
    'recall': recall_score(y_test, y_pred, zero_division = 0),
    'f1': f1_score(y_test, y_pred, zero_division = 0),
    'pr_auc_score': average_precision_score(y_test, y_score)
}

In [37]:
results_df_svm = pd.DataFrame([train_metrics, val_metrics, test_metrics],
                            index = ['Train', 'Validation', 'Test'])
results_df_svm = results_df_svm.T

In [38]:
display(results_df_svm)

,Train,Validation,Test
balanced_accuracy,0.702724,0.537754,0.650000
precision,0.367891,0.209524,0.333333
recall,0.610526,0.340000,0.500000
f1,0.457645,0.257609,0.400000
pr_auc_score,0.601150,0.480631,0.571637
